# 01b — Feature Engineering (Phase A)

**Prerequisite**: Run `notebooks/01_eda.ipynb` fully first. The `df` variable is NOT inherited — this notebook reloads from the saved parquet.

**What happens here**:
1. Reload clean DataFrame from `data/processed/load_metadata.json` + re-runs `load_raw()` to get `df`
2. Parse `violation_type` JSON list → `violation_type_primary` (first atomic type)
3. Derive `is_at_junction = (junction_name != 'No Junction').astype(int8)`
4. Extract temporal features: `hour_of_day`, `day_of_week`, `is_weekend`, `month`
5. Impute `center_code` nulls with mode per `police_station`
6. LabelEncode all categoricals → save `label_encoders.pkl`
7. Save `features_row_level.parquet`

**Files saved**:
- `data/processed/label_encoders.pkl`
- `data/processed/feature_metadata.json`
- `data/processed/features_row_level.parquet`

## Cell 1 — Environment setup
**What this cell does**: Adds project root to `sys.path`.
**Expected output**: `Project root: ...GridLock R2`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\palur\OneDrive\Desktop\GridLock_R2_Transfer


## Cell 2 — Configure loguru
**Expected output**: `Loguru configured.`

In [2]:
import sys
from loguru import logger

logger.remove()
logger.add(
    sys.stdout,
    format='<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}',
    level='DEBUG',
    colorize=False,
)
print('Loguru configured.')

Loguru configured.


## Cell 3 — Reload clean DataFrame
**What this cell does**: Re-runs `load_raw()` to get the clean, deduplicated DataFrame.
This notebook is self-contained — it does not rely on variables from 01_eda.ipynb.

**Expected output**: `Loaded df: (268281, 12)`

In [3]:
from src.data.load import load_raw

CSV_PATH      = PROJECT_ROOT / 'data' / 'raw' / 'jan to may police violation_anonymized791b166.csv'
EVAL_CFG_PATH = PROJECT_ROOT / 'configs' / 'eval.yaml'
FEAT_CFG_PATH = PROJECT_ROOT / 'configs' / 'features.yaml'

df, load_meta = load_raw(
    csv_path=CSV_PATH,
    eval_config_path=EVAL_CFG_PATH,
    save_report=False,   # Already saved in 01_eda.ipynb
)

print(f'Loaded df: {df.shape}')

17:33:59 | INFO     | Reading CSV: 'C:\Users\palur\OneDrive\Desktop\GridLock_R2_Transfer\data\raw\jan to may police violation_anonymized791b166.csv' ...


Reading CSV: 100%|██████████| 1/1 [00:04<00:00,  4.45s/file]

17:34:04 | INFO     | Raw CSV loaded: 298,450 rows × 24 columns | SHA-256: 6e4b7ad2e4c713a6...
17:34:04 | INFO     | Loaded eval config v1.0 from 'C:\Users\palur\OneDrive\Desktop\GridLock_R2_Transfer\configs\eval.yaml'
17:34:04 | INFO     | ─── Schema Validation Start ───────────────────────────────────────
17:34:04 | INFO     | DataFrame shape: 298,450 rows × 24 columns
17:34:04 | INFO     | ✓ All 9 required columns present


17:34:04 | INFO     | ✓ All coordinates within Bengaluru bounding box
17:34:04 | WARNING  | created_datetime: 5 nulls after coerce (≤10 — warning, will be dropped)
17:34:04 | INFO     | ✓ created_datetime range: 2023-11-09 → 2024-04-08 (5 nulls)
17:34:04 | INFO     | ✓ No calendar day gaps in created_datetime
17:34:04 | INFO     | ✓ 'description' confirmed 100% null (expected)
17:34:04 | INFO     | ✓ 'closed_datetime' confirmed 100% null (expected)
17:34:04 | INFO     | ✓ 'action_taken_timestamp' confirmed 100% null (expected)
17:34:04 | WARNING  | Leakage columns present — ensure load.py drops them before features: ['data_sent_to_scita_timestamp', 'modified_datetime', 'validation_status', 'validation_timestamp', 'updated_vehicle_number', 'updated_vehicle_type']
17:34:04 | INFO     | ✓ violation_type parseable as list (500-row sample check)
17:34:05 | INFO     | Split preview — train rows (≤2024-02-29): 226,296 | test rows (≥2024-03-01): 70,311
17:34:05 | INFO     | ✓ Temporal split gu

Casting dtypes:   0%|          | 0/4 [00:00<?, ?col-group/s]

...

17:34:06 | INFO     | ✓ Dtypes cast



Dropping excluded columns: 100%|██████████| 1/1 [00:00<00:00, 387.93op/s]

17:34:06 | INFO     | ✓ Dropped 12 excluded columns: ['description', 'closed_datetime', 'action_taken_timestamp', 'data_sent_to_scita_timestamp', 'modified_datetime', 'validation_status', 'validation_timestamp', 'updated_vehicle_number', 'updated_vehicle_type', 'id', 'vehicle_number', 'location']
17:34:06 | INFO     |   Retained columns (12): ['latitude', 'longitude', 'vehicle_type', 'violation_type', 'offence_code', 'created_datetime', 'device_id', 'created_by_id', 'center_code', 'police_station', 'data_sent_to_scita', 'junction_name']
17:34:06 | INFO     | Null summary (retained columns):
17:34:06 | DEBUG    |   ✓ latitude: 0 nulls
17:34:06 | DEBUG    |   ✓ longitude: 0 nulls
17:34:06 | DEBUG    |   ✓ vehicle_type: 0 nulls
17:34:06 | DEBUG    |   ✓ violation_type: 0 nulls
17:34:06 | DEBUG    |   ✓ offence_code: 0 nulls
17:34:06 | DEBUG    |   ✓ created_datetime: 0 nulls
17:34:06 | DEBUG    |   ✓ device_id: 0 nulls
17:34:06 | DEBUG    |   ✓ created_by_id: 0 nulls
17:34:06 | WARNING  |


Deduplication: 100%|██████████| 2/2 [00:00<00:00,  8.00step/s]

17:34:06 | INFO     | ✓ Deduplication complete: 298,445 → 268,281 rows (30,164 exact duplicates removed)
17:34:06 | INFO     | Split preview → train (≤2024-02-29): 202,968 rows | test (2024-03-01–2024-04-08): 62,583 rows
17:34:06 | INFO     | ─── load_raw() complete: 268,281 rows × 12 columns ───
Loaded df: (268281, 12)


## Cell 4 — Run extract_row_features()
**What this cell does**: Full Phase A feature engineering pipeline.

**Expected output**:
- 5-step progress bar
- `violation_type parsed: 15–20 primary types, 0 parse failures`
- `is_at_junction derived: ~136k junction rows (~50%)`
- `center_code imputed: ~10,120 nulls → 0 remaining`
- 4 LabelEncoder lines
- `Output shape: (268281, 22)`

**Runtime**: ~30–60 seconds (parsing 268k violation_type strings)

In [4]:
from src.data.features import extract_row_features, save_feature_metadata

df_feat, encoders, feat_metadata = extract_row_features(
    df,
    features_config_path=FEAT_CFG_PATH,
    encoder_save_path=PROJECT_ROOT / 'data' / 'processed' / 'label_encoders.pkl',
)
save_feature_metadata(
    feat_metadata,
    PROJECT_ROOT / 'data' / 'processed' / 'feature_metadata.json',
)

print(f'\nOutput shape : {df_feat.shape}')
print(f'New columns  : {feat_metadata["new_columns"]}')

17:34:11 | INFO     | Loaded features config v2.0 from 'C:\Users\palur\OneDrive\Desktop\GridLock_R2_Transfer\configs\features.yaml'
17:34:11 | INFO     | features.yaml hash: 37f0d0873d1526f2...


  Parsing violation_type:  98%|█████████▊| 262396/268281 [00:03<00:00, 65366.42it/s]
                                                                                    

17:34:15 | INFO     | ✓ violation_type parsed: 17 primary types, 0 parse failures → set to 'UNKNOWN'


Derive is_at_junction:  20%|██        | 1/5 [00:04<00:16,  4.15s/step]

17:34:15 | INFO     | ✓ is_at_junction derived: 136,362 junction rows (50.8%)


...

17:34:19 | WARNING  | Some center_code nulls remained after per-station imputation — filled with global mode.
17:34:19 | INFO     | ✓ center_code imputed: 9,510 nulls → 0 remaining


  LabelEncoding: 100%|██████████| 4/4 [00:00<00:00, 10.38it/s]
                                                              

17:34:20 | INFO     |   ✓ Encoded 'violation_type_primary_encoded': 17 unique classes
17:34:20 | INFO     |   ✓ Encoded 'vehicle_type_encoded': 22 unique classes
17:34:20 | INFO     |   ✓ Encoded 'police_station_id': 54 unique classes
17:34:20 | INFO     |   ✓ Encoded 'center_code_encoded': 52 unique classes


Label encoding: 100%|██████████| 5/5 [00:08<00:00,  1.74s/step]

17:34:20 | INFO     | Label encoders saved → 'C:\Users\palur\OneDrive\Desktop\GridLock_R2_Transfer\data\processed\label_encoders.pkl'
17:34:20 | INFO     | ─── extract_row_features() complete: 268,281 rows × 22 cols ───
17:34:20 | INFO     | Feature metadata saved → 'C:\Users\palur\OneDrive\Desktop\GridLock_R2_Transfer\data\processed\feature_metadata.json'

Output shape : (268281, 22)
New columns  : ['violation_type_primary', 'violation_type_primary_encoded', 'is_at_junction', 'hour_of_day', 'day_of_week', 'is_weekend', 'month', 'vehicle_type_encoded', 'police_station_id', 'center_code_encoded']


## Cell 5 — Sanity check: violation_type_primary
**What this cell does**: Confirms violation_type was parsed correctly into readable atomic types.

**Expected output**: Top types are `WRONG PARKING`, `NO PARKING`, etc. — not JSON strings.

In [5]:
print('=== violation_type_primary — top 10 ===')
print(df_feat['violation_type_primary'].value_counts().head(10))

print(f'\nTotal unique primary types: {df_feat["violation_type_primary"].nunique()}')
print(f'Parse failures (UNKNOWN)  : {(df_feat["violation_type_primary"] == "UNKNOWN").sum()}')

=== violation_type_primary — top 10 ===
violation_type_primary
WRONG PARKING                                131744
NO PARKING                                   115964
PARKING IN A MAIN ROAD                        15657
PARKING ON FOOTPATH                            2229
DEFECTIVE NUMBER PLATE                          794
PARKING NEAR ROAD CROSSING                      588
PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC         567
DOUBLE PARKING                                  304
PARKING OTHER THAN BUS STOP                     192
PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS       122
Name: count, dtype: int64

Total unique primary types: 17
Parse failures (UNKNOWN)  : 0


## Cell 6 — Sanity check: is_at_junction
**Expected output**: 0 and 1 only. ~50% junction rows (EDA showed No Junction = 49.5%).

In [6]:
print('=== is_at_junction value counts ===')
vc = df_feat['is_at_junction'].value_counts()
print(vc)
print(f'\nJunction rows: {vc.get(1, 0):,} ({vc.get(1, 0)/len(df_feat)*100:.1f}%)')
print(f'Non-junction : {vc.get(0, 0):,} ({vc.get(0, 0)/len(df_feat)*100:.1f}%)')

=== is_at_junction value counts ===
is_at_junction
1    136362
0    131919
Name: count, dtype: int64

Junction rows: 136,362 (50.8%)
Non-junction : 131,919 (49.2%)


## Cell 7 — Sanity check: temporal features
**Expected output**: `hour_of_day` in [0–23], `month` in [11, 1–4], `is_weekend` only 0/1.

In [7]:
for col in ['hour_of_day', 'day_of_week', 'is_weekend', 'month']:
    print(f'  {col}: min={df_feat[col].min()}, max={df_feat[col].max()}, dtype={df_feat[col].dtype}')

  hour_of_day: min=0, max=23, dtype=int8
  day_of_week: min=0, max=6, dtype=int8
  is_weekend: min=0, max=1, dtype=int8
  month: min=1, max=12, dtype=int8


## Cell 8 — Sanity check: encoded columns
**Expected output**: Integer codes for each categorical. Unique counts match EDA (vehicle_type=22, police_station=54).

In [8]:
for col in ['violation_type_primary_encoded', 'vehicle_type_encoded', 'police_station_id', 'center_code_encoded']:
    n_unique = df_feat[col].nunique()
    sample   = df_feat[col].unique()[:8].tolist()
    print(f'  {col}: {n_unique} unique | sample: {sample}')

  violation_type_primary_encoded: 17 unique | sample: [16, 5, 7, 11, 0, 2, 10, 9]
  vehicle_type_encoded: 22 unique | sample: [1, 16, 17, 13, 11, 3, 21, 6]
  police_station_id: 54 unique | sample: [33, 5, 6, 47, 45, 40, 49, 11]
  center_code_encoded: 52 unique | sample: [51, 44, 17, 21, 6, 4, 22, 10]


## Cell 9 — LabelEncoder classes
**Expected output**: Full class list for each encoder. Verify violation types look right (no JSON strings).

In [9]:
for enc_col, le in encoders.items():
    print(f'\n{enc_col} ({len(le.classes_)} classes):')
    print(f'  {list(le.classes_)}')


violation_type_primary_encoded (17 classes):
  ['DEFECTIVE NUMBER PLATE', 'DEMANDING EXCESS FARE', 'DOUBLE PARKING', 'FAIL TO USE SAFETY BELTS', 'H T V PROHIBITED', 'NO PARKING', 'OBSTRUCTING DRIVER', 'PARKING IN A MAIN ROAD', 'PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC', 'PARKING NEAR ROAD CROSSING', 'PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS', 'PARKING ON FOOTPATH', 'PARKING OPPOSITE TO ANOTHER PARKED VEHICLE', 'PARKING OTHER THAN BUS STOP', 'REFUSE TO GO FOR HIRE', 'WITHOUT SIDE MIRROR', 'WRONG PARKING']

vehicle_type_encoded (22 classes):
  ['BUS (BMTC/KSRTC)', 'CAR', 'FACTORY BUS', 'GOODS AUTO', 'HGV', 'JEEP', 'LGV', 'LORRY/GOODS VEHICLE', 'MAXI-CAB', 'MINI LORRY', 'MOPED', 'MOTOR CYCLE', 'OTHERS', 'PASSENGER AUTO', 'PRIVATE BUS', 'SCHOOL VEHICLE', 'SCOOTER', 'TANKER', 'TEMPO', 'TOURIST BUS', 'TRACTOR', 'VAN']

police_station_id (54 classes):
  ['Adugodi', 'Ashok Nagar', 'Banashankari', 'Banaswadi', 'Basavanagudi', 'Bellandur', 'Byatarayanapura', 'Chamarajpet', 'Chikkabanavara', 'Ch

## Cell 10 — Save features_row_level.parquet
**What this cell does**: Saves `df_feat` as parquet for use by `src/models/clustering.py`.

**Expected output**: File path, shape, and size in MB.

In [10]:
out_path = PROJECT_ROOT / 'data' / 'processed' / 'features_row_level.parquet'
df_feat.to_parquet(out_path, index=False)
print(f'Saved : {out_path}')
print(f'Shape : {df_feat.shape}')
print(f'Size  : {out_path.stat().st_size / 1e6:.1f} MB')

Saved : C:\Users\palur\OneDrive\Desktop\GridLock_R2_Transfer\data\processed\features_row_level.parquet
Shape : (268281, 22)
Size  : 8.3 MB


## Summary

**What was done**:
- `violation_type` parsed → `violation_type_primary` (15–20 atomic types)
- `is_at_junction` derived from `junction_name != 'No Junction'`
- Temporal features extracted from `created_datetime` (UTC)
- `center_code` nulls imputed with mode per `police_station`
- All categoricals LabelEncoded

**What was saved**:
- `data/processed/label_encoders.pkl` — load at inference, never refit
- `data/processed/feature_metadata.json` — stats + features.yaml hash
- `data/processed/features_row_level.parquet` — full row-level feature set

**Next step**: `notebooks/02_cluster_tuning.ipynb` — DBSCAN eps/min_samples grid search to assign `zone_id`

In [11]:
print('=== Phase A Summary ===')
print(f'  Input rows           : {feat_metadata["input_rows"]:,}')
print(f'  Output rows          : {feat_metadata["output_rows"]:,}')
print(f'  Output cols          : {feat_metadata["output_cols"]}')
print(f'  Violation types      : {feat_metadata["violation_type_stats"]["unique_primary_types"]}')
print(f'  Junction rows        : {feat_metadata["is_at_junction_count"]:,} ({feat_metadata["is_at_junction_pct"]}%)')
print(f'  center_code imputed  : {feat_metadata["center_code_imputation"]["imputed_count"]:,}')
print(f'  features.yaml hash   : {feat_metadata["features_yaml_hash"][:16]}...')
print(f'  Saved: data/processed/label_encoders.pkl')
print(f'  Saved: data/processed/feature_metadata.json')
print(f'  Saved: data/processed/features_row_level.parquet')
print(f'  Next notebook: notebooks/02_cluster_tuning.ipynb')

=== Phase A Summary ===
  Input rows           : 268,281
  Output rows          : 268,281
  Output cols          : 22
  Violation types      : 17
  Junction rows        : 136,362 (50.83%)
  center_code imputed  : 9,510
  features.yaml hash   : 37f0d0873d1526f2...
  Saved: data/processed/label_encoders.pkl
  Saved: data/processed/feature_metadata.json
  Saved: data/processed/features_row_level.parquet
  Next notebook: notebooks/02_cluster_tuning.ipynb
